# Cosmic Event Crowdsourcing Platform: Milestone 1 (Review 2)

A Spark pipeline that detects anomalous celestial events in crowdsourced **AAVSO** variable-star
observations, using each observer's computed reliability as an explicit classifier feature.

**Stack running inside this Colab VM:** Hadoop 3.3.6 (HDFS + YARN, pseudo-distributed) · Spark 3.5.9 on YARN ·
Hive 3.1.3 metastore on a Derby network server · Spark SQL · Spark MLlib · PyTorch (deep learning). **Every graph is a Plotly figure**, shown inline and saved as PNG (slides) + interactive HTML. The architecture diagram is drawn with Graphviz.

Run top to bottom with **Runtime → Run all**. Outputs are written to `MyDrive/cosmic/outputs/review2/` and shown inline below.

## 0. Mount Google Drive and get the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = "https://github.com/lowkey2556/cosmic-events.git"
import os
if not os.path.isdir('/content/cosmic/.git'):
    !git clone -q "$REPO_URL" "/content/cosmic"
else:
    !cd "/content/cosmic" && git pull -q
%cd /content/cosmic
!git log --oneline | head -15

### Data folder
`DATA_ROOT` is the Drive folder that holds the AAVSO CSVs. It is searched recursively (`raw/`, `existing/`, and any
sub-folder added later). ZIPs, other files and `DOWNLOAD_LOG/` are ignored. Stars are identified by `auid`, never by file name.

In [ ]:
DATA_ROOT = "/content/drive/MyDrive/cosmic_roja"   # <- contains raw/ (2024) and existing/ (Jan 2025); read recursively
import os
os.environ["COSMIC_DATA_ROOT"] = DATA_ROOT        # every `!python run.py ...` below reads this
assert os.path.isdir(DATA_ROOT), f"Drive folder not found: {DATA_ROOT}"
!find "$DATA_ROOT" -maxdepth 2 -type d | sort

## Step 0: Dataset audit (plain Python, runs before the Big Data stack)
Per file and in total: size in MB, rows, stars, observers and date range. The total size gets a PASS/FAIL
against the 50 MB target, both as stored and after removing observations repeated across overlapping downloads.
Per star: rows, observers, dates, longest continuous coverage and the number of 100-day windows.
Class balance, skipped files and unmatched VSX types. (1 MB = 1,048,576 bytes.)

In [ ]:
!pip -q install pyyaml astropy astroquery
!python run.py audit --data real

## 1. Install the stack
Java 8 comes from apt. Hadoop, Spark, Hive and Derby tarballs are cached in `MyDrive/cosmic/cache/`, so a reinstall after a Colab reset only extracts them.
The first run downloads about 1.5 GB.

In [ ]:
%%time
!bash setup/install_stack.sh

## 2. Start HDFS, YARN, the Derby network server and the Hive metastore

In [ ]:
%%time
!bash setup/start_stack.sh

In [ ]:
# Load the stack's environment (JAVA_HOME, HADOOP_HOME, SPARK_HOME, HIVE_HOME, PATH...) into this kernel,
# so every later `!command` cell can call hdfs, yarn, spark-submit and hive directly.
import os, re
env_file = open('setup/ENV_LOCATION').read().strip()
for line in open(env_file):
    m = re.match(r'export (\w+)=(.*)', line.strip())
    if m:
        os.environ[m.group(1)] = os.path.expandvars(m.group(2))
!java -version 2>&1 | head -1; hadoop version | head -1; spark-submit --version 2>&1 | grep -m1 version; hive --version 2>/dev/null | head -1

## 3. Smoke tests (each must print PASS before the pipeline runs)

In [ ]:
!bash setup/smoke_tests.sh 1   # hdfs dfs put + list

In [ ]:
!bash setup/smoke_tests.sh 2   # YARN ResourceManager up

In [ ]:
!bash setup/smoke_tests.sh 3   # SparkPi submitted to YARN

In [ ]:
!bash setup/smoke_tests.sh 4   # Hive metastore: table created and queried

In [ ]:
!bash setup/smoke_tests.sh 5   # PySpark-on-YARN Hive table read back through Hive

### Optional: open the web UIs
YARN ResourceManager is on 8088, the HDFS NameNode on 9870, and the Spark UI on 4040 while a job runs.

In [ ]:
from google.colab import output
for port in (8088, 9870):
    output.serve_kernel_port_as_window(port)

## 4. Raw data in Drive

In [ ]:
!find "$DATA_ROOT" -type f -iname '*.csv' -not -path '*/DOWNLOAD_LOG/*' | sort | head -100
!ls -la "/content/drive/MyDrive/cosmic/labels" 2>/dev/null || echo "no labels file yet - Step 1 will fetch VSX rows from VizieR" 

## 5. Architecture diagram (slide 3)

In [ ]:
!which dot >/dev/null || apt-get -qq install -y graphviz   # Graphviz draws the diagram
!python run.py architecture --data real

## 6. Step 1: Ingestion and storage
`hdfs dfs -put` to `/cosmic/raw/` → explicit-schema read → Parquet + Snappy partitioned by `obs_year`/`sky_region` →
Hive external table `observations_raw` → partition-pruning test → dataset summary.

In [ ]:
!python run.py ingest --data real

In [ ]:
# The raw data in HDFS, and the partitions as Hive itself sees them
!hdfs dfs -ls -R -h "/cosmic/raw" | head -60
!hdfs dfs -du -s -h "/cosmic/raw" "/cosmic/warehouse/observations_raw" "/cosmic/warehouse/observations_flat"
!hive -S -e "SHOW PARTITIONS cosmic.observations_raw;" 2>/dev/null

## 7. Step 2: Preprocessing in Spark SQL

In [ ]:
!python run.py preprocess --data real

## 8. Step 3: Observer agreement score

In [ ]:
!python run.py score --data real

## 9. Step 4: Features

In [ ]:
!python run.py features --data real

## 10. Step 5: Classification, one fixed seed

In [ ]:
!python run.py classify --data real

## 11. Step 6: K-Means

In [ ]:
!python run.py cluster --data real

In [ ]:
# Hive tables produced by the pipeline (listed through the Hive CLI)
!hive -S -e "USE cosmic; SHOW TABLES;" 2>/dev/null

# Review 2 deliverables (all from the real data above)

In [ ]:
import pandas as pd
import os
import plotly.io as pio
from IPython.display import Image, Markdown, display
OUT = '/content/drive/MyDrive/cosmic/outputs/review2'
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
def csv(name, title=None):
    display(Markdown(f'**{title or name}**')); display(pd.read_csv(f'{OUT}/{name}'))
def fig(name):
    # show a saved Plotly figure inline (interactive); its PNG and HTML copies are next to it in OUT
    path = f'{OUT}/{name}.json'
    if os.path.exists(path):
        pio.read_json(path).show()
    else:
        display(Markdown(f'*{name}: not produced yet*'))
def png(name, width=900):
    display(Image(f'{OUT}/{name}', width=width))
if os.path.exists(f'{OUT}/figure_export_log.csv'):
    log = pd.read_csv(f'{OUT}/figure_export_log.csv')
    bad = log[~log['png'].astype(bool)]
    print('PNG export: all OK' if bad.empty else f'PNG export FAILED for {len(bad)} figures (HTML kept):')
    if not bad.empty: display(bad)

## Slide 1: Dataset

In [ ]:
csv('audit_summary.csv', 'Dataset audit: summary (50 MB check)')
csv('audit_files.csv', 'Dataset audit: per file'); csv('audit_class_balance.csv', 'Class balance')
csv('audit_skipped_files.csv', 'Skipped files (bad header)'); csv('audit_unmatched_vsx_types.csv', 'Unmatched VSX types')
csv('dataset_summary.csv', 'Dataset summary')
csv('rows_per_band.csv', 'Rows per band'); csv('rows_per_type.csv', 'Rows per observation type')
csv('label_counts.csv', 'Stars per label')
fig('fig_obs_per_star')

## Slide 3: Architecture

In [ ]:
png('architecture.png', 1100)   # Graphviz PNG (SVG and DOT source saved next to it)

## Slide 5: Preprocessing results

In [ ]:
csv('preprocess_row_counts.csv', 'Row counts after each step'); fig('fig_preprocessing_funnel')
csv('partition_pruning.csv', 'Partition pruning: partitioned vs unpartitioned'); fig('fig_partition_pruning')
csv('lightcurve_star.csv', 'Light-curve star'); fig('fig_lightcurve_before_after')
csv('observer_score_summary.csv', 'Observer scores: summary'); fig('fig_observer_score_hist')
csv('observer_top_bottom10.csv', 'Top and bottom 10 observers by agreement score')
csv('spearman_score_vs_uncertainty.csv', 'Spearman: score vs self-reported uncertainty')

## Slide 7: Proposed model results

In [ ]:
csv('split_summary.csv', 'Split by star (auid)')
csv('proposed_model_metrics.csv', 'Proposed model (DT + observer features), seed 42')
fig('fig_confusion_proposed')
csv('feature_log.csv', 'Feature log')
csv('kmeans_silhouette.csv', 'K-Means silhouette'); fig('fig_kmeans_silhouette')
csv('kmeans_crosstab_label.csv', 'Cluster vs label'); csv('kmeans_crosstab_type.csv', 'Cluster vs VSX type')

## Slide 8: Comparison with existing classifiers

In [ ]:
csv('model_metrics.csv', 'All models, same split (seed 42); positive class = anomalous')
fig('fig_model_comparison')
fig('fig_confusion_dt_vs_proposed')

**Caveat:** these are preliminary results from **one seed on a small test set**. Milestone 2 adds the 5-seed robustness (mean ± std).

In [ ]:
!ls -la "/content/drive/MyDrive/cosmic/outputs/review2"